# NBA Player & Team ROI Visualizations — 2024-25 Season
Based on the Lautier et al. linear performance model (WRMS / SGV framework).

**Break-even ROI = 1.0×** — above means the player generated more value than he was paid.

---
### Charts included
**Player-level:**
1. Top 10 ROI — All Players
2. Bottom 10 ROI — All Players
3. Top 10 ROI — 65-Game Qualified
4. Bottom 10 ROI — 65-Game Qualified
5. ROI vs. Salary Scatter
6. ROI Distribution (All vs. Qualified)
7. Value Over/Under Contract (Qualified, Top & Bottom 12)
8. Performance Score (Delta) vs. ROI Bubble Chart

**Team-level:**
9. Average Player ROI — All 30 Teams
10. Aggregate Team ROI — All 30 Teams
11. Team Payroll vs. Avg ROI Scatter
12. Team Value Over/Under Contract (Dollar Terms)
13. % of Roster Above Break-even per Team
---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
from scipy.stats import gaussian_kde
import warnings
warnings.filterwarnings('ignore')

# ── Global dark theme ─────────────────────────────────────────────────────────
BG      = '#0f1117'
PANEL   = '#1a1d27'
GRID    = '#2e3040'
TEXT    = '#e0e0e0'
SUBTEXT = '#a0a0b0'
GOLD    = '#f5a623'
GREEN   = '#4ecdc4'
RED     = '#e05252'
BLUE    = '#4a90d9'
PURPLE  = '#9b59b6'

plt.rcParams.update({
    'figure.facecolor': BG,    'axes.facecolor': PANEL,
    'axes.edgecolor':  '#3a3d4d', 'axes.labelcolor': TEXT,
    'axes.titlecolor': '#ffffff', 'axes.titlesize': 13,
    'axes.labelsize':  10,   'xtick.color': SUBTEXT, 'ytick.color': TEXT,
    'xtick.labelsize': 9,    'ytick.labelsize': 9.5,
    'text.color':      TEXT, 'grid.color': GRID,
    'grid.linestyle':  '--', 'grid.alpha': 0.6,
    'legend.facecolor': PANEL, 'legend.edgecolor': '#3a3d4d',
    'legend.labelcolor': TEXT, 'font.family': 'DejaVu Sans',
})

# ── NBA team color map ────────────────────────────────────────────────────────
TEAM_COLORS = {
    'Atlanta Hawks':            '#E03A3E',
    'Boston Celtics':           '#007A33',
    'Brooklyn Nets':            '#778899',
    'Charlotte Hornets':        '#00788C',
    'Chicago Bulls':            '#CE1141',
    'Cleveland Cavaliers':      '#860038',
    'Dallas Mavericks':         '#00538C',
    'Denver Nuggets':           '#FEC524',
    'Detroit Pistons':          '#C8102E',
    'Golden State Warriors':    '#1D428A',
    'Houston Rockets':          '#CE1141',
    'Indiana Pacers':           '#FDBB30',
    'LA Clippers':              '#C8102E',
    'Los Angeles Lakers':       '#552583',
    'Memphis Grizzlies':        '#5D76A9',
    'Miami Heat':               '#98002E',
    'Milwaukee Bucks':          '#00471B',
    'Minnesota Timberwolves':   '#236192',
    'New Orleans Pelicans':     '#0C2340',
    'New York Knicks':          '#006BB6',
    'Oklahoma City Thunder':    '#007AC1',
    'Orlando Magic':            '#0077C0',
    'Philadelphia 76ers':       '#006BB6',
    'Phoenix Suns':             '#E56020',
    'Portland Trail Blazers':   '#E03A3E',
    'Sacramento Kings':         '#5A2D81',
    'San Antonio Spurs':        '#8A8D8F',
    'Toronto Raptors':          '#CE1141',
    'Utah Jazz':                '#002B5C',
    'Washington Wizards':       '#E31837',
}

def tc(team, fallback='#4a90d9'):
    return TEAM_COLORS.get(team, fallback)

def fmt_m(x, pos=None):
    return f'${x:.0f}M'

# ── Load data ─────────────────────────────────────────────────────────────────
from nba_roi.config import ROI_FILE, FIGURES_DIR
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

df   = pd.read_excel(ROI_FILE, sheet_name='ROI Results')
logs = pd.read_excel(ROI_FILE, sheet_name='Game Logs + WRMS')

df = df.rename(columns={
    'Salary ($)':                        'Salary',
    'Avg Performance Score (Delta)':     'Avg_Delta',
    'Avg WRMS Share':                    'Avg_WRMS',
    'Total Cash Flow ($)':               'Total_CF',
    'Value Over/Under Contract ($)':     'Value_Diff',
    'ROI (Total_CF / Salary)':           'ROI',
})

# 65-game qualified subset
df_q = df[df['GAMES_PLAYED'] >= 65].copy()

# ── Build team-level summary ──────────────────────────────────────────────────
team = df.groupby('Team').agg(
    Avg_ROI      = ('ROI',      'mean'),
    Median_ROI   = ('ROI',      'median'),
    Total_CF     = ('Total_CF', 'sum'),
    Total_Salary = ('Salary',   'sum'),
    Players      = ('ROI',      'count'),
    Above_BE     = ('ROI',      lambda x: (x > 1.0).sum()),
).reset_index()
team['Team_ROI']     = team['Total_CF'] / team['Total_Salary']
team['Value_Diff']   = team['Total_CF'] - team['Total_Salary']
team['Pct_Above_BE'] = team['Above_BE'] / team['Players'] * 100

print(f'Players loaded:        {len(df)}')
print(f'65-game qualified:     {len(df_q)}')
print(f'Teams:                 {len(team)}')
print(f'League avg ROI:        {df["ROI"].mean():.3f}x')
print(f'League median ROI:     {df["ROI"].median():.3f}x')

---
## Chart 1 — Top 10 ROI (All Players)
Minimum-salary players with outstanding per-game numbers dominate this list. Games played are shown to provide context — a very high ROI on few games reflects efficiency, not sustained contribution.

In [ ]:
top10 = df.nlargest(10, 'ROI').sort_values('ROI')

fig, ax = plt.subplots(figsize=(12, 6))
fig.patch.set_facecolor(BG)

colors = [tc(t) for t in top10['Team']]
bars   = ax.barh(top10['Player'], top10['ROI'],
                 color=colors, edgecolor=BG, linewidth=0.6, height=0.65)

ax.axvline(1.0, color=GOLD, lw=1.6, ls='--', label='Break-even (1.0x)', zorder=5)

for bar, row in zip(bars, top10.itertuples()):
    ax.text(bar.get_width() + 0.4, bar.get_y() + bar.get_height() / 2,
            f'{row.ROI:.1f}x   {row.GAMES_PLAYED}G  |  ${row.Salary/1e6:.1f}M',
            va='center', ha='left', fontsize=8.5, color=SUBTEXT)

ax.set_xlabel('ROI  (Total Cash Flow / Salary)', labelpad=8)
ax.set_title('Top 10 ROI — All Rostered Players  |  2024-25 NBA Season',
             pad=12, fontweight='bold')
ax.set_xlim(0, top10['ROI'].max() * 1.28)
ax.xaxis.grid(True); ax.set_axisbelow(True)
ax.spines[['top','right','left']].set_visible(False)
ax.legend(loc='lower right', fontsize=9)
fig.tight_layout()
plt.savefig(FIGURES_DIR / 'chart1_top10_all.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

---
## Chart 2 — Bottom 10 ROI (All Players)
Typically max/near-max contracts with missed games or poor per-game production. Negative ROI means the model assigned the player negative value — they actively hurt their team's per-game metrics relative to the average.

In [ ]:
bot10 = df.nsmallest(10, 'ROI').sort_values('ROI', ascending=False)

fig, ax = plt.subplots(figsize=(12, 6))
fig.patch.set_facecolor(BG)

colors = [tc(t) for t in bot10['Team']]
bars   = ax.barh(bot10['Player'], bot10['ROI'],
                 color=colors, edgecolor=BG, linewidth=0.6, height=0.65)

ax.axvline(1.0, color=GOLD, lw=1.6, ls='--', label='Break-even (1.0x)')
ax.axvline(0.0, color=RED,  lw=1.2, ls=':',  label='Zero ROI', alpha=0.8)

for bar, row in zip(bars, bot10.itertuples()):
    x_pos = row.ROI + 0.005 if row.ROI >= 0 else row.ROI - 0.005
    ha    = 'left'           if row.ROI >= 0 else 'right'
    ax.text(x_pos, bar.get_y() + bar.get_height() / 2,
            f'{row.ROI:.3f}x   {row.GAMES_PLAYED}G  |  ${row.Salary/1e6:.1f}M',
            va='center', ha=ha, fontsize=8.5, color=SUBTEXT)

ax.set_xlabel('ROI  (Total Cash Flow / Salary)', labelpad=8)
ax.set_title('Bottom 10 ROI — All Rostered Players  |  2024-25 NBA Season',
             pad=12, fontweight='bold')
ax.xaxis.grid(True); ax.set_axisbelow(True)
ax.spines[['top','right','left']].set_visible(False)
ax.legend(loc='lower right', fontsize=9)
fig.tight_layout()
plt.savefig(FIGURES_DIR / 'chart2_bot10_all.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

---
## Chart 3 — Top 10 ROI (65-Game Qualified)
Players who met the NBA award eligibility threshold. These are the best sustained-value contributors of the season — the list shifts noticeably from Chart 1.

In [ ]:
top10_q = df_q.nlargest(10, 'ROI').sort_values('ROI')

fig, ax = plt.subplots(figsize=(12, 6))
fig.patch.set_facecolor(BG)

colors = [tc(t) for t in top10_q['Team']]
bars   = ax.barh(top10_q['Player'], top10_q['ROI'],
                 color=colors, edgecolor=BG, linewidth=0.6, height=0.65)

ax.axvline(1.0, color=GOLD, lw=1.6, ls='--', label='Break-even (1.0x)')

for bar, row in zip(bars, top10_q.itertuples()):
    ax.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height() / 2,
            f'{row.ROI:.2f}x   {row.GAMES_PLAYED}G  |  ${row.Salary/1e6:.1f}M',
            va='center', ha='left', fontsize=8.5, color=SUBTEXT)

ax.set_xlabel('ROI  (Total Cash Flow / Salary)', labelpad=8)
ax.set_title('Top 10 ROI — 65-Game Qualified Players  |  2024-25 NBA Season',
             pad=12, fontweight='bold')
ax.set_xlim(0, top10_q['ROI'].max() * 1.28)
ax.xaxis.grid(True); ax.set_axisbelow(True)
ax.spines[['top','right','left']].set_visible(False)
ax.legend(loc='lower right', fontsize=9)
fig.text(0.13, 0.005, 'Min. 65 games played (NBA award eligibility threshold)',
         fontsize=8, color='#606070', style='italic')
fig.tight_layout()
plt.savefig(FIGURES_DIR / 'chart3_top10_qualified.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

---
## Chart 4 — Bottom 10 ROI (65-Game Qualified)
Full-season starters who generated the least value relative to their contract. They can't hide behind missed games — these are the most costly under-performers.

In [ ]:
bot10_q = df_q.nsmallest(10, 'ROI').sort_values('ROI', ascending=False)

fig, ax = plt.subplots(figsize=(12, 6))
fig.patch.set_facecolor(BG)

colors = [tc(t) for t in bot10_q['Team']]
bars   = ax.barh(bot10_q['Player'], bot10_q['ROI'],
                 color=colors, edgecolor=BG, linewidth=0.6, height=0.65)

ax.axvline(1.0, color=GOLD, lw=1.6, ls='--', label='Break-even (1.0x)')

for bar, row in zip(bars, bot10_q.itertuples()):
    ax.text(bar.get_width() + 0.003, bar.get_y() + bar.get_height() / 2,
            f'{row.ROI:.3f}x   {row.GAMES_PLAYED}G  |  ${row.Salary/1e6:.1f}M',
            va='center', ha='left', fontsize=8.5, color=SUBTEXT)

ax.set_xlabel('ROI  (Total Cash Flow / Salary)', labelpad=8)
ax.set_title('Bottom 10 ROI — 65-Game Qualified Players  |  2024-25 NBA Season',
             pad=12, fontweight='bold')
ax.set_xlim(0, bot10_q['ROI'].max() * 1.4)
ax.xaxis.grid(True); ax.set_axisbelow(True)
ax.spines[['top','right','left']].set_visible(False)
ax.legend(loc='lower right', fontsize=9)
fig.text(0.13, 0.005, 'Min. 65 games played (NBA award eligibility threshold)',
         fontsize=8, color='#606070', style='italic')
fig.tight_layout()
plt.savefig(FIGURES_DIR / 'chart4_bot10_qualified.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

---
## Chart 5 — ROI vs. Salary Scatter
Log-scale salary reveals the classic value vs. star dynamic. Top-left = best value (low salary, high ROI); bottom-right = worst.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 7))
fig.patch.set_facecolor(BG)

bins        = [0, 2e6, 10e6, 25e6, np.inf]
tier_labels = ['Min/Rookie (<$2M)', 'Mid-level ($2M-$10M)',
               'Starter ($10M-$25M)', 'Max/Near-max (>$25M)']
tier_colors = [GREEN, BLUE, GOLD, RED]

df['Tier'] = pd.cut(df['Salary'], bins=bins, labels=tier_labels)

for tier, color in zip(tier_labels, tier_colors):
    mask = df['Tier'] == tier
    ax.scatter(df.loc[mask,'Salary'], df.loc[mask,'ROI'].clip(upper=20),
               c=color, alpha=0.72, s=40, label=tier, zorder=3, edgecolors='none')

ax.axhline(1.0, color=GOLD, lw=1.6, ls='--', label='Break-even (1.0x)', zorder=4)

highlight = pd.concat([
    df.nlargest(5, 'ROI'), df.nsmallest(4, 'ROI'), df_q.nlargest(5, 'ROI')
]).drop_duplicates('Player')
for _, row in highlight.iterrows():
    ax.annotate(row['Player'].split()[-1],
                xy=(row['Salary'], min(row['ROI'], 20)),
                xytext=(5, 3), textcoords='offset points',
                fontsize=7.5, color='#d0d0e0', alpha=0.9,
                arrowprops=dict(arrowstyle='-', color='#555566', lw=0.5))

ax.set_xscale('log')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(
    lambda x, _: f'${x/1e6:.0f}M' if x >= 1e6 else f'${x/1e3:.0f}K'))
ax.set_xlabel('Salary  (log scale)', labelpad=8)
ax.set_ylabel('ROI  (clipped at 20x)', labelpad=8)
ax.set_title('ROI vs. Salary — All Players  |  2024-25 NBA Season',
             pad=12, fontweight='bold')
ax.grid(True, which='both', alpha=0.25); ax.set_axisbelow(True)
ax.spines[['top','right']].set_visible(False)
ax.legend(loc='upper right', fontsize=8.5, markerscale=1.3)
fig.tight_layout()
plt.savefig(FIGURES_DIR / 'chart5_roi_vs_salary.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

---
## Chart 6 — ROI Distribution: All Players vs. 65-Game Qualified
Histogram + KDE overlay. The qualified cohort has a tighter, more normal distribution — extreme outliers on cheap deals are filtered out.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
fig.patch.set_facecolor(BG)

configs = [
    (df,   'All Rostered Players  (n=468)', 30),
    (df_q, '65-Game Qualified  (n=169)',    12),
]

for ax, (data, title, clip) in zip(axes, configs):
    roi = data['ROI'].clip(upper=clip)
    ax.set_facecolor(PANEL)

    _, bins_arr, patches = ax.hist(roi, bins=40, density=True, edgecolor='none', alpha=0.0)
    _, bins_arr, patches = ax.hist(roi, bins=40, density=True, color=GREEN, edgecolor='none', alpha=0.6)
    for patch, left in zip(patches, bins_arr[:-1]):
        patch.set_facecolor(RED if left < 1.0 else GREEN)
        patch.set_alpha(0.65)

    kde   = gaussian_kde(roi, bw_method=0.3)
    x_rng = np.linspace(roi.min() - 0.5, roi.max() + 0.5, 300)
    ax.plot(x_rng, kde(x_rng), color=GOLD, lw=2.0, label='KDE')

    ax.axvline(1.0,                   color=GOLD,   lw=1.8, ls='--',
               label='Break-even (1.0x)')
    ax.axvline(data['ROI'].median(),  color=PURPLE, lw=1.4, ls=':',
               label=f"Median ({data['ROI'].median():.2f}x)")
    ax.axvline(data['ROI'].mean(),    color='#2ecc71', lw=1.4, ls=':',
               label=f"Mean ({data['ROI'].mean():.2f}x)")

    under = (data['ROI'] < 1.0).sum()
    over  = (data['ROI'] >= 1.0).sum()
    ax.text(0.03, 0.97, f'Under break-even: {under}',
            transform=ax.transAxes, fontsize=8.5, color=RED,   va='top')
    ax.text(0.03, 0.91, f'Over break-even:  {over}',
            transform=ax.transAxes, fontsize=8.5, color=GREEN, va='top')
    if clip < 30:
        ax.text(0.97, 0.97, f'(ROI clipped at {clip}x for display)',
                transform=ax.transAxes, fontsize=7.5, color='#606070',
                va='top', ha='right', style='italic')

    ax.set_xlabel('ROI', labelpad=8)
    ax.set_ylabel('Density', labelpad=8)
    ax.set_title(title, pad=10, fontweight='bold')
    ax.legend(fontsize=8, loc='upper right')
    ax.spines[['top','right']].set_visible(False)
    ax.grid(axis='y', alpha=0.3)

fig.suptitle('ROI Distribution  |  2024-25 NBA Season',
             fontsize=14, fontweight='bold', color='white', y=1.02)
fig.tight_layout()
plt.savefig(FIGURES_DIR / 'chart6_roi_distribution.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

---
## Chart 7 — Value Over/Under Contract: Dollar Terms (65-Game Qualified)
Absolute dollar surplus/deficit for the top and bottom 12 qualified players. Shows whose contracts were the best and worst investments in raw dollar terms.

In [ ]:
combined = pd.concat([
    df_q.nlargest(12, 'Value_Diff'),
    df_q.nsmallest(12, 'Value_Diff')
]).drop_duplicates('Player').sort_values('Value_Diff')

fig, ax = plt.subplots(figsize=(12, 10))
fig.patch.set_facecolor(BG)

bar_colors = [GREEN if v >= 0 else RED for v in combined['Value_Diff']]
bars = ax.barh(combined['Player'], combined['Value_Diff'] / 1e6,
               color=bar_colors, edgecolor=BG, linewidth=0.4, height=0.72)

ax.axvline(0, color=GOLD, lw=1.5, ls='--')

for bar, row in zip(bars, combined.itertuples()):
    val  = row.Value_Diff / 1e6
    sign = '+' if val >= 0 else ''
    off  = 0.5  if val >= 0 else -0.5
    ha   = 'left' if val >= 0 else 'right'
    ax.text(val + off, bar.get_y() + bar.get_height() / 2,
            f'{sign}${val:.1f}M', va='center', ha=ha, fontsize=8.5, color='#d0d0e0')

ax.set_xlabel('Value Over / Under Contract  ($ Millions)', labelpad=8)
ax.set_title('Value Over/Under Contract — 65-Game Qualified  |  2024-25 NBA Season',
             pad=12, fontweight='bold')
ax.xaxis.grid(True, alpha=0.4); ax.set_axisbelow(True)
ax.spines[['top','right','left']].set_visible(False)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(fmt_m))
over_p  = mpatches.Patch(color=GREEN, label='Outperformed contract')
under_p = mpatches.Patch(color=RED,   label='Underperformed contract')
ax.legend(handles=[over_p, under_p], loc='lower right', fontsize=9)
fig.tight_layout()
plt.savefig(FIGURES_DIR / 'chart7_value_diff_qualified.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

---
## Chart 8 — Performance Score (Delta) vs. ROI Bubble Chart
Bubble size = salary; color = games played. Separates per-game quality (Delta) from availability. High Delta + low ROI = injury-prone star. Low Delta + high ROI = available role player on a minimum deal.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 7))
fig.patch.set_facecolor(BG)

plot_df          = df.copy()
plot_df['ROI_c'] = plot_df['ROI'].clip(upper=15)

sc = ax.scatter(
    plot_df['Avg_Delta'], plot_df['ROI_c'],
    c=plot_df['GAMES_PLAYED'], cmap='plasma',
    s=plot_df['Salary'] / 8e4,
    alpha=0.65, edgecolors='none', zorder=3
)

cbar = fig.colorbar(sc, ax=ax, pad=0.01)
cbar.set_label('Games Played', fontsize=9, color=SUBTEXT)
plt.setp(cbar.ax.yaxis.get_ticklabels(), color=SUBTEXT, fontsize=8)

ax.axhline(1.0, color=GOLD,    lw=1.6, ls='--', label='Break-even ROI (1.0x)')
ax.axvline(0.0, color=SUBTEXT, lw=1.0, ls=':',  alpha=0.5, label='Avg Delta = 0')

q_kw = dict(fontsize=8, alpha=0.4, style='italic', color='#8080a0')
ax.text(plot_df['Avg_Delta'].quantile(0.02), 14.0, 'Cheap + Poor',  ha='left', **q_kw)
ax.text(plot_df['Avg_Delta'].quantile(0.75), 14.0, 'Cheap + Elite', ha='left', **q_kw)
ax.text(plot_df['Avg_Delta'].quantile(0.02), 0.15, 'Expensive + Poor',  ha='left', **q_kw)
ax.text(plot_df['Avg_Delta'].quantile(0.75), 0.15, 'Expensive + Elite', ha='left', **q_kw)

highlight = pd.concat([
    df.nlargest(4, 'ROI'), df.nsmallest(3, 'ROI'), df_q.nlargest(4, 'ROI')
]).drop_duplicates('Player')
for _, row in highlight.iterrows():
    ax.annotate(row['Player'].split()[-1],
                xy=(row['Avg_Delta'], min(row['ROI'], 15)),
                xytext=(5, 3), textcoords='offset points',
                fontsize=7.5, color='#d0d0e0',
                arrowprops=dict(arrowstyle='-', color='#555566', lw=0.5))

ax.set_xlabel('Avg Performance Score (Delta)  —  higher = better per-game contributor', labelpad=8)
ax.set_ylabel('ROI  (clipped at 15x)', labelpad=8)
ax.set_title('Performance Score vs. ROI  |  Bubble size = Salary  |  Color = Games Played',
             pad=12, fontweight='bold')
ax.legend(loc='upper left', fontsize=9)
ax.spines[['top','right']].set_visible(False)
ax.grid(alpha=0.2)
fig.tight_layout()
plt.savefig(FIGURES_DIR / 'chart8_delta_vs_roi.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

---
# Team-Level Visualizations

Two team ROI metrics are used:
- **Avg ROI** — mean of individual player ROIs. Sensitive to high-ROI role players on cheap deals.
- **Aggregate Team ROI** — total roster cash flow / total payroll. Treats the roster as one investment; large overpaid stars drag this down more than Avg ROI.

> Note: Because WRMS redistributes value across the whole league (not within teams), aggregate team ROI values will cluster well below 1.0× — this is expected and reflects the league-wide redistribution mechanic.

---
## Chart 9 — Average Player ROI by Team (All 30 Teams)
Ranked by the mean ROI across all rostered players. Rewards teams with multiple efficient role players on cheap deals.

In [ ]:
team_avg = team.sort_values('Avg_ROI', ascending=True)

fig, ax = plt.subplots(figsize=(12, 10))
fig.patch.set_facecolor(BG)

colors = [tc(t) for t in team_avg['Team']]
bars   = ax.barh(team_avg['Team'], team_avg['Avg_ROI'],
                 color=colors, edgecolor=BG, linewidth=0.4, height=0.72)

ax.axvline(1.0, color=GOLD, lw=1.8, ls='--', label='Break-even (1.0x)', zorder=5)
league_mean = df['ROI'].mean()
ax.axvline(league_mean, color=BLUE, lw=1.4, ls=':', alpha=0.8,
           label=f'League mean ({league_mean:.2f}x)', zorder=5)

for bar, row in zip(bars, team_avg.itertuples()):
    ax.text(bar.get_width() + 0.03, bar.get_y() + bar.get_height() / 2,
            f'{row.Avg_ROI:.2f}x', va='center', ha='left', fontsize=8, color=SUBTEXT)

ax.set_xlabel('Average Player ROI', labelpad=8)
ax.set_title('Average Player ROI by Team — All 30 Teams  |  2024-25 NBA Season',
             pad=12, fontweight='bold')
ax.set_xlim(0, team_avg['Avg_ROI'].max() * 1.18)
ax.xaxis.grid(True); ax.set_axisbelow(True)
ax.spines[['top','right','left']].set_visible(False)
ax.legend(loc='lower right', fontsize=9)
fig.tight_layout()
plt.savefig(FIGURES_DIR / 'chart9_team_avg_roi.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

---
## Chart 10 — Aggregate Team ROI (Total Cash Flow / Total Payroll)
The truest team-level efficiency measure. Color gradient from red (lowest) to green (highest). Payroll annotation helps contextualize — a high-spending team needs much more production to score well.

In [ ]:
team_agg = team.sort_values('Team_ROI', ascending=True)

fig, ax = plt.subplots(figsize=(12, 10))
fig.patch.set_facecolor(BG)

norm   = plt.Normalize(team_agg['Team_ROI'].min(), team_agg['Team_ROI'].max())
cmap   = plt.cm.RdYlGn
colors = [cmap(norm(v)) for v in team_agg['Team_ROI']]

bars = ax.barh(team_agg['Team'], team_agg['Team_ROI'],
               color=colors, edgecolor=BG, linewidth=0.4, height=0.72)

league_agg_roi = df['Total_CF'].sum() / df['Salary'].sum()
ax.axvline(league_agg_roi, color=GOLD, lw=1.8, ls='--',
           label=f'League aggregate ROI ({league_agg_roi:.3f}x)', zorder=5)

for bar, row in zip(bars, team_agg.itertuples()):
    ax.text(bar.get_width() + 0.003, bar.get_y() + bar.get_height() / 2,
            f'{row.Team_ROI:.3f}x   ${row.Total_Salary/1e6:.0f}M payroll',
            va='center', ha='left', fontsize=8, color=SUBTEXT)

ax.set_xlabel('Team ROI  (Total Roster Cash Flow / Total Payroll)', labelpad=8)
ax.set_title('Aggregate Team ROI — All 30 Teams  |  2024-25 NBA Season',
             pad=12, fontweight='bold')
ax.set_xlim(0, team_agg['Team_ROI'].max() * 1.30)
ax.xaxis.grid(True); ax.set_axisbelow(True)
ax.spines[['top','right','left']].set_visible(False)
ax.legend(loc='lower right', fontsize=9)

sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, pad=0.01, shrink=0.4, aspect=20)
cbar.set_label('Aggregate ROI', fontsize=8, color=SUBTEXT)
plt.setp(cbar.ax.yaxis.get_ticklabels(), color=SUBTEXT, fontsize=7.5)

fig.tight_layout()
plt.savefig(FIGURES_DIR / 'chart10_team_agg_roi.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

---
## Chart 11 — Team Payroll vs. Average Player ROI Scatter
Reveals which teams got the most efficiency per dollar. Quadrant labels describe each corner of the spend/efficiency spectrum.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))
fig.patch.set_facecolor(BG)

scatter_colors = [tc(t) for t in team['Team']]
ax.scatter(team['Total_Salary'] / 1e6, team['Avg_ROI'],
           c=scatter_colors, s=130, zorder=3,
           edgecolors='white', linewidths=0.4, alpha=0.92)

med_sal = team['Total_Salary'].median() / 1e6
med_roi = team['Avg_ROI'].median()
ax.axvline(med_sal, color=SUBTEXT, lw=1.0, ls=':', alpha=0.5,
           label=f'Median payroll (${med_sal:.0f}M)')
ax.axhline(med_roi, color=SUBTEXT, lw=1.0, ls=':', alpha=0.5,
           label=f'Median avg ROI ({med_roi:.2f}x)')
ax.axhline(1.0, color=GOLD, lw=1.5, ls='--', label='Break-even ROI (1.0x)', alpha=0.7)

for _, row in team.iterrows():
    short = row['Team'].replace('Los Angeles ', 'LA ').split()[-1]
    ax.annotate(short,
                xy=(row['Total_Salary'] / 1e6, row['Avg_ROI']),
                xytext=(4, 4), textcoords='offset points',
                fontsize=7.5, color='#d0d0e0', alpha=0.88)

q_kw = dict(fontsize=8.5, alpha=0.35, style='italic', color='#9090a0')
ax.text(100,  team['Avg_ROI'].max() * 0.97, 'Low spend + High efficiency',  ha='left', va='top', **q_kw)
ax.text(195,  team['Avg_ROI'].max() * 0.97, 'High spend + High efficiency', ha='left', va='top', **q_kw)
ax.text(100,  med_roi * 0.45,               'Low spend + Low efficiency',   ha='left', **q_kw)
ax.text(195,  med_roi * 0.45,               'High spend + Low efficiency',  ha='left', **q_kw)

ax.set_xlabel('Total Team Payroll  ($ Millions)', labelpad=8)
ax.set_ylabel('Average Player ROI', labelpad=8)
ax.set_title('Team Payroll vs. Average Player ROI  |  2024-25 NBA Season',
             pad=12, fontweight='bold')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(fmt_m))
ax.grid(alpha=0.2); ax.set_axisbelow(True)
ax.spines[['top','right']].set_visible(False)
ax.legend(loc='upper right', fontsize=8.5)
fig.tight_layout()
plt.savefig(FIGURES_DIR / 'chart11_team_payroll_vs_roi.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

---
## Chart 12 — Team Value Over/Under Contract (Dollar Terms)
The dollar gap between total cash flows generated by each roster and what was paid. This captures both performance and availability — injured expensive players create large negative gaps.

In [ ]:
team_vd = team.sort_values('Value_Diff', ascending=True)

fig, ax = plt.subplots(figsize=(12, 10))
fig.patch.set_facecolor(BG)

bar_colors = [GREEN if v >= 0 else RED for v in team_vd['Value_Diff']]
bars = ax.barh(team_vd['Team'], team_vd['Value_Diff'] / 1e6,
               color=bar_colors, edgecolor=BG, linewidth=0.4, height=0.72)

ax.axvline(0, color=GOLD, lw=1.5, ls='--', label='Break-even ($0)')

for bar, row in zip(bars, team_vd.itertuples()):
    val  = row.Value_Diff / 1e6
    sign = '+' if val >= 0 else ''
    off  =  0.8 if val >= 0 else -0.8
    ha   = 'left' if val >= 0 else 'right'
    ax.text(val + off, bar.get_y() + bar.get_height() / 2,
            f'{sign}${val:.0f}M', va='center', ha=ha, fontsize=8, color=SUBTEXT)

ax.set_xlabel('Value Over / Under Contract  ($ Millions)', labelpad=8)
ax.set_title('Team Value Over/Under Contract — All 30 Teams  |  2024-25 NBA Season',
             pad=12, fontweight='bold')
ax.xaxis.grid(True, alpha=0.4); ax.set_axisbelow(True)
ax.spines[['top','right','left']].set_visible(False)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(fmt_m))
over_p  = mpatches.Patch(color=GREEN, label='Roster outperformed payroll')
under_p = mpatches.Patch(color=RED,   label='Roster underperformed payroll')
ax.legend(handles=[over_p, under_p], loc='lower right', fontsize=9)
fig.tight_layout()
plt.savefig(FIGURES_DIR / 'chart12_team_value_diff.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

---
## Chart 13 — % of Roster Above Break-even per Team
What share of each team's players exceeded ROI = 1.0×? Teams with more players above break-even have better depth efficiency — their role players are justifying their contracts.

In [ ]:
team_pct = team.sort_values('Pct_Above_BE', ascending=True)

fig, ax = plt.subplots(figsize=(12, 10))
fig.patch.set_facecolor(BG)

norm_pct   = plt.Normalize(0, 100)
cmap_pct   = plt.cm.RdYlGn
colors_pct = [cmap_pct(norm_pct(v)) for v in team_pct['Pct_Above_BE']]

bars = ax.barh(team_pct['Team'], team_pct['Pct_Above_BE'],
               color=colors_pct, edgecolor=BG, linewidth=0.4, height=0.72)

ax.axvline(50, color=GOLD, lw=1.5, ls='--', label='50% of roster above break-even')

for bar, row in zip(bars, team_pct.itertuples()):
    ax.text(bar.get_width() + 0.8, bar.get_y() + bar.get_height() / 2,
            f'{row.Pct_Above_BE:.0f}%  ({row.Above_BE}/{row.Players})',
            va='center', ha='left', fontsize=8, color=SUBTEXT)

ax.set_xlabel('% of Roster with ROI > 1.0x', labelpad=8)
ax.set_title('% of Roster Above Break-even per Team  |  2024-25 NBA Season',
             pad=12, fontweight='bold')
ax.set_xlim(0, 105)
ax.xaxis.set_major_formatter(mticker.PercentFormatter())
ax.xaxis.grid(True); ax.set_axisbelow(True)
ax.spines[['top','right','left']].set_visible(False)
ax.legend(loc='lower right', fontsize=9)
fig.tight_layout()
plt.savefig(FIGURES_DIR / 'chart13_team_pct_above_be.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()

---
## Summary Statistics

In [ ]:
print('=' * 60)
print('PLAYER ROI SUMMARY  (All Players)')
print('=' * 60)
print(f"  Players analysed:      {len(df)}")
print(f"  Mean ROI:              {df['ROI'].mean():.3f}x")
print(f"  Median ROI:            {df['ROI'].median():.3f}x")
print(f"  Above break-even:      {(df['ROI'] > 1).sum()} ({(df['ROI'] > 1).mean():.1%})")
print(f"  Best ROI:   {df.loc[df['ROI'].idxmax(), 'Player']:<25}  {df['ROI'].max():.2f}x")
print(f"  Worst ROI:  {df.loc[df['ROI'].idxmin(), 'Player']:<25}  {df['ROI'].min():.3f}x")

print()
print('=' * 60)
print('PLAYER ROI SUMMARY  (65-Game Qualified)')
print('=' * 60)
print(f"  Players analysed:      {len(df_q)}")
print(f"  Mean ROI:              {df_q['ROI'].mean():.3f}x")
print(f"  Median ROI:            {df_q['ROI'].median():.3f}x")
print(f"  Above break-even:      {(df_q['ROI'] > 1).sum()} ({(df_q['ROI'] > 1).mean():.1%})")
print(f"  Best ROI:   {df_q.loc[df_q['ROI'].idxmax(), 'Player']:<25}  {df_q['ROI'].max():.2f}x")
print(f"  Worst ROI:  {df_q.loc[df_q['ROI'].idxmin(), 'Player']:<25}  {df_q['ROI'].min():.3f}x")

print()
print('=' * 60)
print('TEAM ROI SUMMARY')
print('=' * 60)
best_avg  = team.loc[team['Avg_ROI'].idxmax()]
worst_avg = team.loc[team['Avg_ROI'].idxmin()]
best_agg  = team.loc[team['Team_ROI'].idxmax()]
worst_agg = team.loc[team['Team_ROI'].idxmin()]
most_be   = team.loc[team['Pct_Above_BE'].idxmax()]
least_be  = team.loc[team['Pct_Above_BE'].idxmin()]
print(f"  Best avg ROI:    {best_avg['Team']:<30} {best_avg['Avg_ROI']:.2f}x")
print(f"  Worst avg ROI:   {worst_avg['Team']:<30} {worst_avg['Avg_ROI']:.2f}x")
print(f"  Best agg ROI:    {best_agg['Team']:<30} {best_agg['Team_ROI']:.3f}x")
print(f"  Worst agg ROI:   {worst_agg['Team']:<30} {worst_agg['Team_ROI']:.3f}x")
print(f"  Most BE players: {most_be['Team']:<30} {most_be['Pct_Above_BE']:.0f}% ({most_be['Above_BE']}/{most_be['Players']})")
print(f"  Fewest BE:       {least_be['Team']:<30} {least_be['Pct_Above_BE']:.0f}% ({least_be['Above_BE']}/{least_be['Players']})")

print()
print('Charts saved to outputs/figures/:')
for i, name in enumerate([
    'chart1_top10_all.png',           'chart2_bot10_all.png',
    'chart3_top10_qualified.png',     'chart4_bot10_qualified.png',
    'chart5_roi_vs_salary.png',       'chart6_roi_distribution.png',
    'chart7_value_diff_qualified.png','chart8_delta_vs_roi.png',
    'chart9_team_avg_roi.png',        'chart10_team_agg_roi.png',
    'chart11_team_payroll_vs_roi.png','chart12_team_value_diff.png',
    'chart13_team_pct_above_be.png',
], 1):
    print(f'  {i:>2}. {name}')